# Stranger Things — Sentiment & Polarity by Season
Stacked bar = Sentiment % (Positive / Neutral / Negative) | Line = Avg Polarity Score

In [ ]:
# Install: pip install vaderSentiment matplotlib pandas
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## Step 1: Load Data

In [ ]:
df = pd.read_csv("combined_data.csv")

## Step 2: Run VADER Sentiment Analysis

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Neutral", 0.0
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive", score
    elif score <= -0.05:
        return "Negative", score
    else:
        return "Neutral", score

print("Running sentiment & polarity analysis...")
results = df["text"].apply(get_sentiment)
df["sentiment"] = results.apply(lambda x: x[0])
df["compound"]  = results.apply(lambda x: x[1])
print("Done!")
print(df["sentiment"].value_counts())

## Step 3: Aggregate by Season

In [ ]:
order  = ["Season 1", "Season 2", "Season 3", "Season 4", "Season 5"]
labels = ["S1", "S2", "S3", "S4", "S5"]

counts = (
    df.groupby(["season", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reindex(order)
)
for col in ["Positive", "Neutral", "Negative"]:
    if col not in counts.columns:
        counts[col] = 0
pct = counts.div(counts.sum(axis=1), axis=0) * 100

polarity = (
    df.groupby("season")["compound"]
    .mean()
    .reindex(order)
    .round(4)
)

print("\nSentiment % by season:")
print(pct.round(1))
print("\nAvg polarity by season:")
print(polarity)

## Step 4: Plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor("#1a1a2e")
ax1.set_facecolor("#1a1a2e")

x     = np.arange(len(order))
width = 0.5
colors = {"Positive": "#4CAF50", "Neutral": "#9E9E9E", "Negative": "#E63946"}

bottoms = np.zeros(len(order))
for sentiment in ["Positive", "Neutral", "Negative"]:
    values = pct[sentiment].values
    ax1.bar(x, values, width, bottom=bottoms,
            color=colors[sentiment], alpha=0.85, label=sentiment, zorder=2)
    for i, (val, bot) in enumerate(zip(values, bottoms)):
        if val > 3:
            ax1.text(x[i], bot + val / 2, f"{val:.1f}%",
                     ha="center", va="center", fontsize=9, fontweight="bold", color="white")
    bottoms += values

ax1.set_ylim(0, 100)
ax1.set_ylabel("Sentiment Percentage (%)", color="white", fontsize=11)
ax1.tick_params(axis="y", colors="white")
ax1.set_xticks(x)
ax1.set_xticklabels(labels, color="white", fontsize=11)
ax1.tick_params(axis="x", colors="white")
ax1.spines[:].set_visible(False)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

ax2 = ax1.twinx()
ax2.plot(x, polarity.values, color="#FFB703", linewidth=2.5,
         linestyle="--", marker="o", markersize=8, zorder=4)
for i, val in enumerate(polarity.values):
    ax2.text(x[i], val + 0.015, f"{val:.3f}",
             ha="center", fontsize=9, color="#FFB703", fontweight="bold")
ax2.set_ylim(0, 0.6)
ax2.set_ylabel("Avg Polarity Score (-1 to +1)", color="white", fontsize=11)
ax2.tick_params(axis="y", colors="white")
ax2.spines[:].set_visible(False)
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))

legend_elements = [
    Patch(facecolor="#4CAF50", label="Positive"),
    Patch(facecolor="#9E9E9E", label="Neutral"),
    Patch(facecolor="#E63946", label="Negative"),
    Line2D([0], [0], color="#FFB703", linestyle="--", marker="o", markersize=6, label="Avg Polarity Score"),
]
ax1.legend(handles=legend_elements, loc="upper center",
           bbox_to_anchor=(0.5, -0.08), ncol=4,
           facecolor="#2d2d44", edgecolor="none", labelcolor="white", fontsize=10)

plt.title("Stranger Things — Sentiment & Polarity by Season (IMDb + Reddit)",
          color="white", fontsize=14, fontweight="bold", pad=15)
plt.subplots_adjust(bottom=0.15)
plt.savefig("sentiment_polarity.png", dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
print("\nSaved: sentiment_polarity.png")
plt.show()